# 23장 · 재정렬 · 검색 후보와 답변에 넣을 문단 고르기

재정렬은 1차 검색이 가져온 후보들에 다시 점수를 매기거나 순서를 붙이는 단계입니다. 검색하지 않은 문서는 재정렬 결과에도 추가되지 않습니다.

## 같은 후보의 순서 바꾸기

`sorted(scores, key=scores.get, reverse=True)`는 후보 ID를 점수가 높은 순서로 나열합니다. 아래의 두 점수는 순서를 읽기 위한 예시입니다. 실제 재정렬 함수는 뒤에서 질문과 후보 원문을 모델에 함께 보냅니다.

In [ ]:
scores = {"A": 0.2, "B": 0.9}
ordered = sorted(scores, key=scores.get, reverse=True)
print(ordered[:1])

['B']


두 점수를 미리 준 예제입니다. 큰 점수부터 정렬해서 B가 첫 후보가 됐습니다. 뒤의 실습에서는 수업 함수 `rerank(question, candidates)`가 실제 모델에 후보의 순서를 요청합니다.

## 응용 실습 · 설명서 후보 재정렬하기

## 실습 준비

전체 실습 묶음을 풀고 노트북을 엽니다. 같은 폴더에서 시작해 자료 묶음의 `pre-mid/documents`를 읽습니다. 공용 준비 파일에 뒤에서 사용할 함수도 있으므로 **아래 셀에서 가져오는 이름**부터 살펴보세요. 처음 한 번 커널에서 `%pip install pypdf langchain-openai langchain-core chromadb python-dotenv`를 실행하고 커널을 재시작하세요. 작업 폴더의 .env에 OPENAI_API_KEY를 둡니다. 첫 색인 생성과 새 질문 임베딩은 API를 호출합니다. 화면은 `%pip install streamlit`로 준비합니다.

In [1]:
from pathlib import Path
import sys

# 압축을 푼 폴더 또는 그 아래에서 실습 자료의 위치를 찾습니다.
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pre-mid/documents/rag/rag_lab.py").is_file()), None)
if root is None:
    raise FileNotFoundError("전체 실습 묶음을 풀고 pre-mid 폴더 옆에서 실행하세요.")
rag_dir = root / "pre-mid/documents/rag"
sys.path.insert(0, str(rag_dir))

## retrieve와 rerank의 결과 순서 비교하기

첫 검색에서 네 쪽을 가져와도 실제 답변에는 상위 두 쪽만 보낼 수 있습니다. ‘후보 네 개 중 정답은 3위’이면 재정렬로 정답을 두 쪽 안에 넣을 여지가 있습니다. ‘후보 네 개에 정답이 없음’이라면 네 쪽을 어떻게 섞어도 정답을 얻지 못합니다. 이때는 질문 표현·후보 수·색인부터 살펴야 합니다.

`retrieve(index, question, k=4)`로 후보를 고른 뒤, `rerank(question, candidates)`가 같은 후보의 본문과 ID를 모델에 보내 순서를 다시 정합니다. 이 교재 함수는 응답 ID 목록에 누락이나 중복이 있는지 검사하고 문제가 있으면 처음 순서를 돌려줍니다. 첫 검색에서 빠진 쪽은 재정렬해도 생기지 않습니다. 답변에 실제로 들어가는 범위는 다음 단계의 `[:2]`입니다.

In [2]:
from rag_lab import open_index, retrieve, rerank, answer_from_documents
index = open_index()
question = "2027년 3월 15일 A형으로 계속 녹음하면 파일은 몇 분마다 나뉘나요?"
candidates = retrieve(index, question, k=4)
ordered = rerank(question, candidates)
print("기본", [x["id"] for x in candidates])
print("재정렬", [x["id"] for x in ordered])

기본 ['recorder-a-1.0.pdf#p2', 'recorder-a-1.1.pdf#p2', 'recorder-b-1.0.pdf#p2', 'recorder-a-1.0.pdf#p1']
재정렬 ['recorder-a-1.0.pdf#p2', 'recorder-a-1.0.pdf#p1', 'recorder-a-1.1.pdf#p2', 'recorder-b-1.0.pdf#p2']


수업의 재정렬 함수는 모델에게 **현재 후보의 ID와 본문**을 주고 순서만 돌려받습니다. 새 ID를 만들거나 일부를 빼면 원래 순서를 사용합니다. 모델을 한 번 더 부르므로 대기 시간과 호출량이 늘어납니다. 이 비용이 어느 질문에서 가치가 있는지 판단하려면 첫 후보를 고정해 비교해야 합니다.

In [3]:
# 두 답 모두 같은 질문과 같은 개수의 문단을 받습니다.
print("기본 검색 답", answer_from_documents(question, candidates[:2]))
print("재정렬 답", answer_from_documents(question, ordered[:2]))

기본 검색 답 2027년 3월 15일 A형으로 계속 녹음하면 파일은 30분마다 나뉩니다. 이는 1.0판이 2027년 3월 1일부터 3월 31일까지 적용되며, 이 판에서는 한 번에 길게 녹음하면 30분마다 새 파일이 열린다고 명시되어 있기 때문입니다.[recorder-a-1.0.pdf#p2]
재정렬 답 2027년 3월 15일 A형으로 계속 녹음하면 파일은 30분마다 나뉩니다. "한 번에 길게 녹음하면 30분마다 새 파일이 열립니다."라는 문장이 2027년 3월 1일부터 3월 31일까지 적용되는 1.0판 설명서에 명시되어 있습니다.[recorder-a-1.0.pdf#p2]


## Streamlit에서 검색 방식만 바꾸어 두 결과 비교하기

다음 셀은 12장의 채팅창에 검색 방식 선택과 근거 펼쳐 보기를 붙인 앱 파일을 만듭니다. 질문을 보냈을 때만 검색하고, 위젯만 바뀌면 저장한 결과를 다시 그립니다. 같은 질문을 다른 방식으로 **다시 제출**해 비교하세요. 셀을 실행한 작업 폴더에서 터미널에 `python -m streamlit run compare_search.py`를 입력합니다.

In [4]:
# 노트북 셀을 실행하면 현재 폴더에 화면 파일을 생성합니다.
app_code = r'''import streamlit as st
import sys
from pathlib import Path
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pre-mid/documents/rag").is_dir())
sys.path.insert(0, str(root / "pre-mid/documents/rag"))
from rag_lab import open_index, retrieve, rerank, answer_from_documents

st.title("들소리 설명서 검색 비교")
mode = st.radio("검색 방식", ["기본", "재정렬"], horizontal=True)
if "runs" not in st.session_state:
    st.session_state.runs = []
question = st.chat_input("기종과 사용 시점을 넣어 질문하세요.")
if question:
    candidates = retrieve(open_index(), question, k=4)
    ranked = rerank(question, candidates) if mode == "재정렬" else candidates
    answer = answer_from_documents(question, ranked[:2])
    st.session_state.runs.append((question, mode, answer, candidates, ranked))
for q, method, answer, before, after in st.session_state.runs:
    st.chat_message("user").write(q)
    st.chat_message("assistant").write(f"**{method}**\n\n{answer}")
    with st.expander("처음 찾은 후보와 답변에 보낸 근거"):
        st.write([r["id"] for r in before])
        for row in after[:2]:
            st.write(row["id"], row["text"])
'''
Path("compare_search.py").write_text(app_code, encoding="utf-8")
print("compare_search.py 생성")

compare_search.py 생성


## 답만 읽을 때 가려지는 검색 실패

`keys.index(gold)`는 정답 ID가 처음 나온 위치를 0부터 돌려줍니다. 화면에 표시할 순위는 여기에 1을 더합니다. 후보에 정답이 없으면 순위를 구할 수 없어 `None`으로 남깁니다. 3위가 1위로 올라오는 경우와 처음부터 후보 밖인 경우를 구분하세요.

화면에는 처음 검색한 후보들의 ID를 보여 주고, 답변에 보낸 최종 두 쪽의 원문을 펼칩니다. 평가자는 ‘원래 가져온 후보’, ‘모델이 받은 자료’, ‘응답이 실제로 인용한 자료’를 별도 항목으로 기록해야 합니다. 근거가 후보 4위에 있었다는 말은 생성 모델이 그것을 읽었다는 뜻이 아닙니다.

In [5]:
gold = "recorder-a-1.0.pdf#p2"
for name, rows in [("기본", candidates), ("재정렬", ordered)]:
    keys = [r["id"] for r in rows]
    rank = keys.index(gold) + 1 if gold in keys else None
    print(name, "첫 후보 안의 순위", rank, "답변에 전달됨", gold in keys[:2])

기본 첫 후보 안의 순위 1 답변에 전달됨 True
재정렬 첫 후보 안의 순위 1 답변에 전달됨 True


두 실행 모두 첫 두 쪽에 정답이 있다면 이 질문으로는 재정렬의 이점이 확인되지 않습니다. 반대로 정답이 들어갔는데 인용이 다른 페이지라면 생성 단계까지 조사합니다. 개선 보고에 유리한 사례만 고르지 않도록 같은 질문 두 개를 먼저 정해 둡니다.

오래 걸린다는 체감은 모델·네트워크 상태에 따라 달라질 수 있습니다. 같은 환경에서 같은 질문을 여러 번 보내며 첫 토큰까지 기다린 시간인지 최종 답이 끝난 시간인지 정해 측정하세요. 작은 실습에서 한 번 잰 결과는 교실 전체의 응답 시간이라고 일반화하기 어렵습니다.

## 실습 · 정답이 없는 후보를 재정렬하기

D03의 정답인 A형 1.0판 2쪽을 후보에서 빼고 순서를 다시 정합니다. 결과에 정답이 다시 생길 수 있는지 실험하세요. 원래 후보와 정답을 제거한 후보를 각각 이용해 동료에게 재정렬을 쓸지, 먼저 후보를 넓힐지 설명합니다.

In [6]:
omit_id = "recorder-a-1.0.pdf#p2"
# candidates에서 이 ID를 제외하고 남은 후보만 rerank에 넘기세요.
# 전후의 ID는 집합으로, 순서는 리스트로 비교하세요.

<details class="notebook-answer"><summary>모범 풀이와 판정 기준</summary><pre><code>missing = [r for r in candidates if r[&quot;id&quot;] != &quot;recorder-a-1.0.pdf#p2&quot;]
print([r[&quot;id&quot;] for r in missing])
result = rerank(question, missing)
print([r[&quot;id&quot;] for r in result])
print(&quot;후보 유지&quot;, {r[&quot;id&quot;] for r in missing} == {r[&quot;id&quot;] for r in result})</code></pre><p>재정렬은 빠진 정답을 만들어 내지 못합니다. 원래부터 답이 후보에 없었다면 k나 검색 조건을 먼저 고칩니다. 원래 후보에서 순서가 좋아지지 않았다면 그 결과 역시 그대로 기록하세요.</p><p>집필 때 저장한 재정렬 후의 ID와 집합 비교 결과입니다. 위 코드에서는 재정렬 전 순서도 첫 줄에 출력합니다.</p><pre>[&#x27;recorder-a-1.0.pdf#p1&#x27;, &#x27;recorder-a-1.1.pdf#p2&#x27;, &#x27;recorder-b-1.0.pdf#p2&#x27;]
후보 유지 True</pre></details>

## 독립 실습 · 재정렬 호출을 채택할지 결정하기

D01과 D03을 기본/재정렬 두 방식으로 각각 보냅니다. 처음 검색한 후보 집합을 고정하고 답변에 보낼 수는 2로 맞춥니다. 매번 첫 두 쪽에 정답이 있었는지, 실제 답이 그 쪽을 인용했는지 확인합니다. 한 질문이라도 나빠졌다면 그 사례의 원문과 후보를 붙이세요. 두 방식이 같았던 사례도 검토에 포함합니다.

| 질문 | 첫 후보 | 기본 상위 2개 | 재정렬 상위 2개 | 인용 판정 | 이 질문에 기능을 쓸지 |
|---|---|---|---|---|---|
| D01 | | | | | |
| D03 | | | | | |

<details class="notebook-answer"><summary>판정 예시와 해설</summary><p>D01의 정답은 A형 1.1판 1쪽, D03의 정답은 A형 1.0판 2쪽입니다. 예를 들어 D03의 첫 후보에 3월판이 없으면 재정렬을 추가해도 그 판본이 나오지 않습니다. 처음 네 쪽에는 들어왔지만 두 쪽 안에 없던 정답이 재정렬 뒤 포함됐고 실제 답도 30분을 인용했다면 개선된 한 사례로 기록합니다. 이것만으로 모든 질문에 채택하지는 않고 다른 질문과 추가 호출 비용을 함께 봅니다.</p></details>

## 다음 수업에 가져갈 것

한 질문에 대해 기본/재정렬의 후보·실제 인용·응답 대기 경험을 함께 남기세요. 다음 장에서는 자료나 설정을 바꾸면서도 이전 문서가 답에 섞이지 않는 화면을 완성합니다.